# Homework

In [2]:
%pip install gitsource

/workspaces/llm_zoomcamp_01/llm-zoomcamp-2026-code/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [10]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

len(documents)

72

In [13]:
from minsearch import Index

index = Index(
    text_fields = ['content'], #all fields that are useful for search
    keyword_fields = ['filename'], #fields that are useful for filtering
)

index.fit(documents)

NameError: name 'documents' is not defined

In [12]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "content": 0.5},
    num_results=5
)

search_results

NameError: name 'index' is not defined

In [11]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client
)



NameError: name 'index' is not defined

In [6]:
question = 'How does the agentic loop keep calling the model until it stops?'
assistant.response = assistant.rag(question)
print(assistant.response)
print("input tokens:", assistant.usage)

The agentic loop keeps calling the model in a `while True` loop. Each turn it:

1. sends the full message history to the model,
2. checks the response for any `function_call` items,
3. runs those tools and appends the results,
4. repeats.

It stops when the model returns a response with no function calls:

```python
if has_function_calls == False:
    break
```

So the loop continues until the model gives a final answer without asking for any more tools.
input tokens: ResponseUsage(input_tokens=7125, input_tokens_details=InputTokensDetails(cached_tokens=6912), output_tokens=109, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7234)


In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size = 2000, step = 1000)
len(chunks)

295

In [10]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [11]:
from minsearch import Index

chunk_index = Index(
    text_fields = ['content'], #all fields that are useful for search
    keyword_fields = ['filename']
)

chunk_index.fit(chunks)

In [12]:
assistant = RAGBase(
    index=chunk_index,
    llm_client=openai_client
)

result = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(result)

The loop keeps calling the model with a `while True` loop and stops when the model returns no `function_call` items.

Specifically:

- Each turn, it calls `responses.create(...)`
- It checks the output for any `function_call`
- If there was at least one, it runs the tool and loops again
- If there were **no function calls**, it breaks out of the loop

So the exit condition is:

```python
if has_function_calls == False:
    break
```

That means the model decides when it’s done, and the code just keeps looping until it stops asking for tools.


In [14]:
print("input tokens:", assistant.usage)

input tokens: ResponseUsage(input_tokens=2308, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=131, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2439)


In [23]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes — in most cases you can still join, but it depends on the course’s enrollment policy and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Usually the key things are:\n- whether registration is still open\n- if there’s a waitlist\n- whether you need instructor approval\n- whether there are any prerequisites\n\nIf you’d like, send me:\n- the course name or link\n- where it’s hosted\n- and your deadline if you have one\n\nI can help you draft a message to the instructor or check the likely next steps.'

In [1]:
def search(query: str) -> dict[str, str]:
    """Search the FAQ database for entries matching the given query."""
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"filename": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [26]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course after it has started"}', call_id='call_KMb3n3cDVQWLnTR4JlYLZ3fp', name='search', type='function_call', id='fc_075d68c0b0334d19006a37ff433d6c81a09c97bff4950a9fff', namespace=None, status='completed')]

In [30]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [3]:
!uv add toyaikit
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

Resolved 126 packages in 38ms
Checked 122 packages in 65ms


In [6]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [7]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [15]:
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

In [18]:
instructions = instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
""".strip()

In [21]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [22]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received
